# 00 - Patient Selection
Downloads the numeric files of all ICU stays in MIMIC-IV Waveform Database v0.1.0. and selects 30 patients for our project

## 1. Setup
Load the libraries and define where the data is online (BASE) and where it is locally (DATA_DIR)

In [1]:
import os
import urllib.request
import pandas as pd

BASE = "https://physionet.org/files/mimic4wdb/0.1.0/"
DATA_DIR = os.path.join("..", "dataset", "mimic4wdb")
os.makedirs(DATA_DIR, exist_ok=True)

## 2. List of patients
Read the dataset's main RECORDS file: official list of all 198 patients

In [2]:
patients = pd.read_csv(BASE + "RECORDS", header=None, names=["path"])["path"].tolist()
print(len(patients), "patients")
print(patients[:3])

198 patients
['waves/p100/p10014354/', 'waves/p100/p10019003/', 'waves/p100/p10020306/']


## 3. Link each patient to their ICU stay(s) 
Each patient folder has its own RECORDS file with the stay(s) number(s). We need the stay number because the data file is named after the stay and not the patient. 
Larger pauses prevent server errors (502) and retries handle temporary server errors (502).

In [5]:
import time

def fetch_text(url, tries=5):
    for attempt in range(1, tries + 1):
        try:
            return urllib.request.urlopen(url, timeout=30).read().decode()
        except Exception as e:
            print(f"  attempt {attempt} failed ({e}), waiting {5 * attempt} s...")
            time.sleep(5 * attempt)
    raise RuntimeError(f"Gave up on {url}")

records = []
for i, p in enumerate(patients, start=1):
    text = fetch_text(BASE + p + "RECORDS")
    for line in text.split():
        records.append(p + line)
    time.sleep(0.2)
    if i % 20 == 0:
        print(f"{i}/{len(patients)} patients checked")

print(len(records), "records")
print(records[:3])

20/198 patients checked
40/198 patients checked
60/198 patients checked
80/198 patients checked
100/198 patients checked
120/198 patients checked
140/198 patients checked
160/198 patients checked
180/198 patients checked
200 records
['waves/p100/p10014354/81739927/81739927', 'waves/p100/p10019003/87033314/87033314', 'waves/p100/p10020306/83404654/83404654']


## 4. Download the numerics files
We download only the **numerics** (`<stay>n.csv.gz`): the vital signs the monitor derives from the waveforms. They match our question (deterioration over minutes to hours) and are much smaller (stay 83404654: 7.2 MB vs. ~143 MB of waveforms). Decisions can be seen in D1 of 'decisions.md'.

The download is resumable (existing files are skipped), safe (`.part` until complete), retries server errors (502), and records stays without numerics (404), which are excluded.

In [7]:
import urllib.error

REDOWNLOAD = False   # change to True to download EVERYTHING again

def download(url, target, tries=5):
    for attempt in range(1, tries + 1):
        try:
            urllib.request.urlretrieve(url, target + ".part")
            os.replace(target + ".part", target)
            return "ok"
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return "missing"
            print(f"  attempt {attempt} failed ({e}), waiting {5 * attempt} s...")
            time.sleep(5 * attempt)
        except Exception as e:
            print(f"  attempt {attempt} failed ({e}), waiting {5 * attempt} s...")
            time.sleep(5 * attempt)
    return "failed"

skipped, downloaded, missing, failed = [], [], [], []

for i, rec in enumerate(records, start=1):
    parts = rec.split("/")
    patient, record_id = parts[2], parts[-1]
    filename = f"{patient}_{record_id}n.csv.gz"
    target = os.path.join(DATA_DIR, filename)

    if os.path.exists(target) and not REDOWNLOAD:
        skipped.append(rec)
    else:
        result = download(BASE + rec + "n.csv.gz", target)
        if result == "ok":
            downloaded.append(rec)
        elif result == "missing":
            missing.append(rec)
        else:
            failed.append(rec)
        time.sleep(0.2)

    if i % 20 == 0:
        print(f"{i}/{len(records)} records checked")

print()
print(len(skipped), "already there (skipped)")
print(len(downloaded), "newly downloaded")
print(len(missing), "have no numerics file (404)")
print(len(failed), "failed:", failed)

20/200 records checked
40/200 records checked
60/200 records checked
80/200 records checked
100/200 records checked
120/200 records checked
  attempt 1 failed (HTTP Error 502: Bad Gateway), waiting 5 s...
140/200 records checked
160/200 records checked
180/200 records checked
200/200 records checked

56 already there (skipped)
138 newly downloaded
6 have no numerics file (404)
0 failed: []


## 5. Check the download
Count the downloaded numerics files and their total size, as a check.
**Result:** 194 files, 579 MB in total.
- 200 ICU stays in the dataset; 6 have no numerics file (404) and are excluded, leaving 194.
- The numerics are ~4.5% of the full dataset (579 MB vs. 12.8 GB, which is mostly waveforms)

In [8]:
files = [f for f in os.listdir(DATA_DIR) if f.endswith("n.csv.gz")]
total_mb = sum(os.path.getsize(os.path.join(DATA_DIR, f)) for f in files) / 1e6
print(len(files), "files,", round(total_mb), "MB in total")

194 files, 579 MB in total


## 6. Look inside a numerics file
Example: patient p10020306, stay 83404654.

**Findings:**
- The file has 21 columns: `time` plus 20 monitor values.
- We use/focus 5 vital signs: heart rate `HR [bpm]`, systolic blood pressure `NBPs [mmHg]`, diastolic blood pressure `NBPd [mmHg]`, respiratory rate `RR [rpm]`, oxigen saturation `SpO2 [%]` (see D6).
- Each row is a moment where the monitor recorded at least one value, so most cells are empty (`NaN`). The cuff blood pressure is only measured every few minutes.
- `time` is in ticks of ~1 ms (999.56 ticks per second, from the record header). Rows are roughly 1 second apart.
- The rows are **not sorted by time** (e.g. 0 → −4549 → 672), so we must sort them before any time-based step.

In [2]:
example = os.path.join(DATA_DIR, "p10020306_83404654n.csv.gz")
df = pd.read_csv(example)

print(df.shape)
print(list(df.columns))
df.head()

(756823, 21)
['time', 'ABPd [mmHg]', 'ABPm [mmHg]', 'ABPs [mmHg]', 'DeltaQTc [msec]', 'HR [bpm]', 'NBPd [mmHg]', 'NBPm [mmHg]', 'NBPs [mmHg]', 'PVC [/min]', 'Pulse (ABP) [bpm]', 'Pulse (NBP) [bpm]', 'Pulse (SpO2) [bpm]', 'QT [msec]', 'QT-HR [bpm]', 'QTc [msec]', 'RR [rpm]', 'ST-III [mm]', 'ST-V [mm]', 'SpO2 [%]', 'btbHR [bpm]']


,time,ABPd [mmHg],ABPm [mmHg],ABPs [mmHg],DeltaQTc [msec],HR [bpm],NBPd [mmHg],NBPm [mmHg],NBPs [mmHg],PVC [/min],...,Pulse (NBP) [bpm],Pulse (SpO2) [bpm],QT [msec],QT-HR [bpm],QTc [msec],RR [rpm],ST-III [mm],ST-V [mm],SpO2 [%],btbHR [bpm]
0,0,NaN,NaN,NaN,NaN,NaN,77.0,91.0,122.0,NaN,...,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-4549,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86.64
2,672,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,19.0,NaN,NaN,NaN,NaN
3,1696,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN
4,2720,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,21.0,NaN,NaN,NaN,NaN


## 7. How often is each vital sign measured? (one patient)
For patient p10020306 (stay 83404654), we count the measurements of our 5 vital signs, the typical (median) gap between them, and the longest gap.

**Result:**
- HR, RR and SpO2 are measured about **every second** (~265,000 values each); their longest gap is ~24 minutes.
- The cuff blood pressure (NBPs/NBPd) has only ~370 values, and a **25.8-hour gap** without any cuff measurement.
- NBPd has fewer values than NBPs (353 vs. 385): some cuff readings have no diastolic value.

The cuff is the limiting signal, and a long stay is not automatically a complete one. This is why we need a coverage criterion (**Section 9**).

In [4]:
signals = ["HR [bpm]", "RR [rpm]", "SpO2 [%]", "NBPs [mmHg]", "NBPd [mmHg]"]

rows = []
for s in signals:
    t = df.loc[df[s].notna(), "time"].sort_values()
    gaps_s = t.diff() / 999.56
    rows.append({
        "signal": s,
        "measurements": len(t),
        "median gap (s)": round(gaps_s.median(), 1),
        "longest gap (h)": round(gaps_s.max() / 3600, 1),
    })

pd.DataFrame(rows)

,signal,measurements,median gap (s),longest gap (h)
0,HR [bpm],266438,1.0,0.4
1,RR [rpm],264753,1.0,0.4
2,SpO2 [%],264797,1.0,0.4
3,NBPs [mmHg],385,97.3,25.8
4,NBPd [mmHg],353,112.7,25.8


### 7.1 Zoom in: the cuff blood pressure (NBP)

**Result:** mean gap 11.9 min, median 1.6 min, max 1548 min (25.8 h).
- The mean is pulled up by the one 26-hour gap (an outlier), so it is misleading.
- A median of 1.6 min (and a minimum of 5 s) is too short for a real cuff. Possibly the monitor repeats values → see open questions.

In [3]:
nbp = df[["time", "NBPs [mmHg]"]].dropna().sort_values("time")
gaps_min = nbp["time"].diff() / 999.56 / 60
print(len(nbp), "cuff measurements")
print(gaps_min.describe())

385 cuff measurements
count     384.000000
mean       11.915910
std       112.314560
min         0.085371
25%         0.597596
50%         1.622047
75%         3.500207
max      1548.457321
Name: time, dtype: float64
